### MODULE 5 — Flask Backend API 

### Goal
Build a Flask backend that:
- Accepts product input (currently by product_name)
- Generates AI material recommendations
- Calculates sustainability & environment scores
- Uses ML models to predict cost and CO2
- Loads data from PostgreSQL
- Returns clean JSON output to frontend

### 1) Installed required backend packages

#### Activated virtual environment
PowerShell:
```powershell
& C:\Users\varsh\OneDrive\Desktop\EcoPackAI\.venv\Scripts\Activate.ps1

Installed Flask + CORS
pip install flask flask-cors


Verified Flask installation:

python -c "import flask; print(flask.__version__)"


Output example:

Flask 3.1.2

Installed PostgreSQL driver
pip install psycopg2-binary

### 2) Started Flask server

From project root:

python -m backend.app


Expected terminal output:

Running on http://127.0.0.1:5000

Debug mode: on

### 3) Created Flask endpoints (API routes)
Health endpoint (GET)

URL:

http://127.0.0.1:5000/api/health

Test command:

Invoke-RestMethod -Uri "http://127.0.0.1:5000/api/health"


Expected output:

{ "status": "ok" }

Recommendation endpoint (POST)

URL:

http://127.0.0.1:5000/api/recommend

Test command (PowerShell):

$body = @{ product_name = "Smartphone" } | ConvertTo-Json
$res = Invoke-RestMethod -Uri "http://127.0.0.1:5000/api/recommend" -Method Post -Body $body -ContentType "application/json"
$res | ConvertTo-Json -Depth 10


Sample output (example):

{
  "product": {
    "product_category": "Electronics",
    "product_name": "Smartphone",
    "product_weight_kg": 0.22
  },
  "recommendations": [
    {
      "rank": 1,
      "material_name": "Single-wall corrugated cardboard",
      "pred_cost_inr": 38.80,
      "pred_co2_kg": 2.648,
      "recyclability_percent": 92.0,
      "biodegradability_score": 9.0,
      "suitability_score": 0.7560,
      "environment_score": 77.96
    }
  ]
}

### 4) Error handling proof
Error: Wrong endpoint method (GET on POST endpoint)

If I go to:

http://127.0.0.1:5000/api/recommend
 (in browser)

I get:

405 Method Not Allowed
Because /api/recommend must be POST.

Error: Product not found (example)

Command:

$body = @{ product_name = "abc" } | ConvertTo-Json
Invoke-RestMethod -Uri "http://127.0.0.1:5000/api/recommend" -Method Post -Body $body -ContentType "application/json"


Output:

{ "error": "Product 'abc' not found" }

Helpful try/catch (to show error message cleanly)
try {
  $body = @{ product_name = "abc" } | ConvertTo-Json
  Invoke-RestMethod -Uri "http://127.0.0.1:5000/api/recommend" -Method Post -Body $body -ContentType "application/json"
} catch {
  $_.ErrorDetails.Message
}

### 5) PostgreSQL integration proof
Error faced: Password authentication failed

Example error:

{
  "error": "Database load failed: password authentication failed for user \"postgres\""
}


Fix done:

Updated password in backend/db.py

Retested API → it worked

After fix I got proper output:

product + recommendations returned

### 6) ML inference integrated into Flask
ML model files used (saved in /models)

models/rf_cost_model.pkl

models/xgb_co2_model.pkl

models/scaler.pkl

What happens in backend now

Backend loads models using joblib

For each recommended material row:

Predicts cost using RandomForest model

Predicts CO2 using XGBoost model (with scaling)

Returns predictions in JSON as:

pred_cost_inr

pred_co2_kg

Reason:
Rule-based cost/co2 (raw columns) is static.
ML predictions simulate “AI forecast” which is more realistic and scalable.

### 7) Environment score computation

Backend also computes:

environment_score (combined sustainability measure based on recyclability, biodegradability, and predicted CO2)

This is shown:

in API JSON response

in UI results table + top recommendation card

### 8) Restarting Flask

When I change backend/app.py:

Flask auto restarts (debug mode)

If not:

Stop server: CTRL + C

Restart:

python -m backend.app